# Step 6 — Add Phenotype Metadata

## The problem this step solves

You have an integrated atlas of 204,883 cells with cluster assignments and embeddings. But right now, every cell only knows: which library it came from, how many genes it expressed, and which Leiden cluster it is in. It does not know:

- Was this animal obese or lean?
- Was it exercising or sedentary?
- Which tissue was it collected from?
- What day was the tissue dissected?

Without this context, the atlas is a map with no labels. You can see clusters but cannot ask the questions the paper was designed to answer: *which cell types change when a mouse exercises? Which pathways does obesity dysregulate in fat stem cells?*

This step solves that by joining sample-level experimental metadata onto every cell. The join is straightforward — one row per sample in the phenotype table, broadcast to every cell in that sample — but the consequences are significant: after this step, every single-cell analysis can be conditioned on diet, exercise, or tissue.

## Why this matters for the Yang et al. paper

The paper's four experimental groups — SC (standard chow sedentary), TC (standard chow trained), SH (HFD sedentary), TH (HFD trained) — are the entire basis for differential expression. Without the `pheno` column added in this step, there is no way to compare exercise vs. sedentary, or to identify the "rescue" effect (TH vs. SH) that is the paper's central therapeutic question.

The metadata also includes `tissue_collection_day` and `tissue_collection_time`. These are tracked as potential technical confounders — if all HFD samples happened to be dissected on the same day, apparent diet effects could be confounded with collection batch. The paper confirmed these were not significant confounders (Figure S1F), but only because they were recorded and checked.

## Why metadata is kept separate until now

Phenotype annotations come from animal records (cage notes, body weight measurements, exercise logs), not from sequencing. The CellRanger output knows nothing about diet or exercise — it only knows the library ID. Keeping metadata separate until this step makes the pipeline modular: you can reprocess the sequencing data without touching phenotype records, and vice versa. It also makes the join explicit and auditable rather than embedded silently in earlier steps.

The join logic: every cell has a `sample_ID` in its `.obs`. The phenotype table has one row per sample. This step does a left join — every cell keeps all its existing annotations and gains the phenotype columns from its parent sample.

> **ML analogy:** This is exactly feature joining in a relational data pipeline. You have a `cells` table and a `samples` table. The cell knows its `sample_id` foreign key. This step does the join to bring in the sample-level features (condition, tissue, collection metadata) that enable group-level queries.

In [ ]:
import pandas as pd
import scanpy as sc
from pathlib import Path

## Configuration

- `object_path` — path to the combined `.h5ad` from Step 5. The file is overwritten in place after metadata is added, matching the R behavior (`qsave(temp.data, object_path)`).
- `sample_key` — the `.obs` column that holds the sample identifier used to look up rows in `pheno_df`. Must match the index of `pheno_df`. In the Step 3/5 pipeline this is typically `sample_ID` or `library_ID`.
- `pheno_df` — a DataFrame with one row per sample and one column per phenotype variable. The index must contain the sample identifiers that appear in `adata.obs[sample_key]`.

In [ ]:
object_path = Path("rdata/all_tissues_combined.h5ad")
sample_key  = "sample_ID"   # .obs column to join on; equivalent to orig.ident in R

# Phenotype table — replace with your actual metadata file or DataFrame.
# Index must match the values in adata.obs[sample_key].
# Typical sources: a CSV produced from animal records, or a shared lab metadata sheet.
pheno_df = pd.DataFrame({
    "tissue":             {"lib_001": "scWAT", "lib_002": "scWAT", "lib_003": "vWAT",  "lib_004": "SkM"},
    "diet":               {"lib_001": "HFD",   "lib_002": "NCD",   "lib_003": "HFD",   "lib_004": "NCD"},
    "intervention_group": {"lib_001": "EX",    "lib_002": "SED",   "lib_003": "EX",    "lib_004": "SED"},
    "sample_name":        {"lib_001": "Mouse1_scWAT", "lib_002": "Mouse2_scWAT",
                           "lib_003": "Mouse1_vWAT",  "lib_004": "Mouse2_SkM"},
})
# Alternatively, load from a file:
# pheno_df = pd.read_csv("phenotype_metadata.csv", index_col=0)

## Load the combined object

In [ ]:
adata = sc.read_h5ad(object_path)
print(f"Loaded: {adata.n_obs:,} cells x {adata.n_vars:,} genes")
print(f"Existing .obs columns: {adata.obs.columns.tolist()}")

## Validate the join key

Before joining, confirm that every sample identifier in `.obs` has a matching row in `pheno_df`. Any sample present in the object but absent from `pheno_df` will silently receive `NaN` for all metadata columns — a failure that would produce missing groups in downstream differential expression and confuse any plot colored by condition.

This check makes explicit the implicit behavior of R's `match()`, which returns `NA` for unmatched entries without raising an error.

In [ ]:
if sample_key not in adata.obs.columns:
    raise KeyError(
        f"sample_key='{sample_key}' not found in adata.obs. "
        f"Available columns: {adata.obs.columns.tolist()}"
    )

samples_in_object = set(adata.obs[sample_key].unique())
samples_in_pheno  = set(pheno_df.index)

missing_from_pheno = samples_in_object - samples_in_pheno
extra_in_pheno     = samples_in_pheno  - samples_in_object

if missing_from_pheno:
    print(f"WARNING: {len(missing_from_pheno)} sample(s) in the object have no "
          f"metadata row and will receive NaN:\n  {sorted(missing_from_pheno)}")
else:
    print(f"All {len(samples_in_object)} samples have matching metadata rows.")

if extra_in_pheno:
    print(f"Note: {len(extra_in_pheno)} pheno_df row(s) have no matching cells "
          f"and will be ignored:\n  {sorted(extra_in_pheno)}")

## Join metadata onto cells

The R code does:
```r
temp.data@meta.data <- cbind(
  temp.data@meta.data,
  pheno_df[match(temp.data$orig.ident, rownames(pheno_df)), ]
)
```

This is a **left join**: every cell keeps all its existing annotations and gains the columns from `pheno_df` by looking up its sample identifier. Cells from the same sample all receive identical values for the phenotype columns — the sample-level annotation is broadcast to every individual cell.

The Python equivalent uses `pd.merge` with `how="left"` to preserve the exact cell order. Any pheno columns that already exist in `.obs` from a previous run are dropped first to avoid duplicate-suffix columns.

In [ ]:
# Drop any pheno columns already present in .obs to avoid duplicates on re-run
existing_pheno_cols = [c for c in pheno_df.columns if c in adata.obs.columns]
if existing_pheno_cols:
    print(f"Overwriting existing columns: {existing_pheno_cols}")
    adata.obs = adata.obs.drop(columns=existing_pheno_cols)

# Left join: broadcast sample-level phenotype onto every cell
adata.obs = adata.obs.merge(
    pheno_df,
    left_on=sample_key,
    right_index=True,
    how="left",
)

print(f"Metadata columns added: {pheno_df.columns.tolist()}")
print(f"Updated .obs columns:   {adata.obs.columns.tolist()}")
adata.obs.head()

## Verify the result

Confirm the join was correct by checking cell counts per condition. The distribution should reflect the study design — for example, roughly equal numbers of cells per diet group within each tissue (allowing for biological variation in cell yield). Any unexpected imbalance is worth investigating before proceeding to differential expression, since highly unequal group sizes reduce statistical power.

NaN values in phenotype columns indicate samples that were in the object but not in `pheno_df` — these must be resolved before Step 7.

In [ ]:
for col in pheno_df.columns:
    if col in adata.obs.columns:
        print(f"\nCell counts by {col}:")
        print(adata.obs[col].value_counts().to_string())

nan_counts = adata.obs[pheno_df.columns].isna().sum()
if nan_counts.any():
    print("\nWARNING — NaN cells per column (unmatched samples):")
    print(nan_counts[nan_counts > 0].to_string())
else:
    print("\nNo NaN values — all cells matched successfully.")

## Save (overwrite in place)

The R script saves back to the same path (`qsave(temp.data, object_path)`), overwriting the Step 5 output. The same behavior is reproduced here.

After this step, the object contains everything needed for downstream analysis in Step 7:

- Raw counts in `layers["counts"]`
- Normalized expression in `.X`
- Embeddings: `X_pca`, `X_pca_integrated`, `X_tsne`, `X_umap`
- Cluster labels: `leiden`, `umap_dbscan`
- Per-cell QC metrics: `total_counts`, `n_genes_by_counts`, `pct_counts_mt`
- Per-cell phenotype: tissue, diet, intervention group, sample name

Having all of this in a single object is what enables the kinds of queries Step 7 needs to ask — e.g., "what genes are differentially expressed in adipocytes between HFD-exercise and HFD-sedentary mice?", which requires simultaneously knowing cell type (from clustering), condition (from `intervention_group`), and diet (from `diet`).

In [ ]:
adata.write_h5ad(object_path)
print(f"Saved (in place): {object_path}")
print(f"Final object: {adata.n_obs:,} cells x {adata.n_vars:,} genes")